# Generate Stock Reasoning using RAG

This notebook:
1. Downloads Siri's stock CSV from GCS
2. Uses RAG system to generate investment reasoning
3. Adds reasoning as a new column
4. Uploads enhanced CSV back to GCS in model_output/ directory


In [1]:
# Imports
import os
import sys
from pathlib import Path
from io import BytesIO
import time
import json
from typing import Dict, Any, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv
from google.cloud import storage
from google.oauth2 import service_account
from tqdm import tqdm

# LangChain imports
from langchain_google_vertexai import ChatVertexAI
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage

# Add parent directory to path to import rag_helpers
sys.path.insert(0, str(Path().absolute().parent))
from rag_files.rag_helpers import get_rag_connection, get_chroma_db, get_embedder

# Load environment variables
load_dotenv(override=True)


C:\Users\eilke\anaconda3\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


True

In [2]:
# Configuration
GCS_BUCKET_NAME = "fin-data-bucket-115"
GCS_CHROMADB_BUCKET = os.getenv("GCS_BUCKET_NAME", "stock-busters-chroma-bucket")
INPUT_CSV = "model_output/combined_quantamental_hybrid_with_factors_and_backtest.csv"
OUTPUT_CSV = "model_output/combined_quantamental_hybrid_with_factors_and_backtest_with_reasoning.csv"
CREDENTIALS_PATH = "../secrets/stock-busters-service-account.json"

# Processing parameters
SAMPLE_SIZE = None  # Set to a number (e.g., 10) for testing, None for all stocks
MAX_WORKERS = 20  # Number of parallel workers

print("Configuration loaded")


Configuration loaded


In [3]:
# Setup Credentials
def setup_credentials():
    """Set up GCP credentials for Vertex AI and GCS."""
    if os.path.exists(CREDENTIALS_PATH):
        creds_file = os.path.abspath(CREDENTIALS_PATH)
    elif os.path.exists("secrets/stock-busters-service-account.json"):
        creds_file = os.path.abspath("secrets/stock-busters-service-account.json")
    else:
        raise FileNotFoundError(
            f"Credentials file not found. Expected at {CREDENTIALS_PATH} or secrets/stock-busters-service-account.json"
        )
    
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = creds_file
    credentials = service_account.Credentials.from_service_account_file(creds_file)
    return credentials

credentials = setup_credentials()
print("✓ Credentials loaded")


✓ Credentials loaded


In [4]:
# GCS Helper Functions
def list_gcs_files(bucket_name: str, prefix: str = "") -> list:
    """List files in GCS bucket with optional prefix."""
    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)
        blobs = list(bucket.list_blobs(prefix=prefix))
        return [blob.name for blob in blobs]
    except Exception as e:
        print(f"Error listing files in GCS: {e}")
        return []

def download_csv_from_gcs(file_name: str, bucket_name: str = GCS_BUCKET_NAME) -> pd.DataFrame:
    """Download CSV file from GCS bucket."""
    print(f"Downloading {file_name} from GCS bucket {bucket_name}...")
    
    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(file_name)
        
        if not blob.exists():
            print(f"File not found at specified path. Searching for '{file_name.split('/')[-1]}' in bucket...")
            filename_only = file_name.split('/')[-1]
            all_files = list_gcs_files(bucket_name)
            matching_files = [f for f in all_files if filename_only in f]
            if matching_files:
                model_output_files = [f for f in matching_files if 'model_output/' in f]
                if model_output_files:
                    file_name = model_output_files[0]
                    print(f"✓ Found in model_output folder: {file_name}")
                else:
                    file_name = matching_files[0]
                    print(f"⚠ Using first match: {file_name}")
                blob = bucket.blob(file_name)
            else:
                raise FileNotFoundError(f"File {filename_only} not found in bucket {bucket_name}")
        
        csv_bytes = blob.download_as_bytes()
        df = pd.read_csv(BytesIO(csv_bytes))
        
        print(f"Successfully loaded {file_name}")
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        
        return df
    except Exception as e:
        print(f"Error downloading CSV from GCS: {e}")
        raise

def upload_csv_to_gcs(df: pd.DataFrame, gcs_path: str, bucket_name: str = GCS_BUCKET_NAME):
    """Upload DataFrame as CSV to GCS bucket."""
    print(f"Uploading CSV to GCS: {gcs_path}...")
    
    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(gcs_path)
        
        csv_buffer = BytesIO()
        df.to_csv(csv_buffer, index=False)
        csv_buffer.seek(0)
        
        blob.upload_from_file(csv_buffer, content_type="text/csv")
        
        print(f"Successfully uploaded to gs://{bucket_name}/{gcs_path}")
    except Exception as e:
        print(f"Error uploading CSV to GCS: {e}")
        raise

print("GCS helper functions defined")


GCS helper functions defined


In [5]:
# Download CSV from GCS (if not already downloaded in main execution)
# This cell can be run separately for testing, or skip to Cell 13 for full execution
if 'df' not in locals():
    print("[1/5] Downloading CSV from GCS...")
    df = download_csv_from_gcs(INPUT_CSV, GCS_BUCKET_NAME)
    print("✓ CSV downloaded")
else:
    print("CSV already loaded (df variable exists)")


[1/5] Downloading CSV from GCS...


Successfully loaded model_output/combined_quantamental_hybrid_with_factors_and_backtest.csv
Shape: (390, 40)
Columns: ['symbol', 'pred_prob_next_month', 'signal', 'Hybrid_Score', 'Fundamental_Score', 'Technical_Score', 'Hybrid_Rank', 'Hybrid_CS_Pct', 'H_Score Recommendation', 'date', 'roe', 'roic', 'peRatio', 'freeCashFlowYield', 'debtToEquity', 'currentRatio', 'dividendYield', 'earningsYield', 'payoutRatio', 'cashPerShare', 'revenuePerShare', 'return_1m', 'ema_12', 'ema_26', 'macd', 'macd_signal', 'macd_hist', 'RSI_14', 'volatility_21d', 'n_periods', 'avg_fwd_1m_ret', 'vol_1m', 'sharpe_1m_annual', 'max_drawdown', 'hit_rate_pos', 'hit_rate_vs_sp500', 'cagr', 'equity_chart_path', 'sector', 'industry']
✓ CSV downloaded


In [6]:
# Setup RAG System - Part 1: Connect to ChromaDB
print("[2/5] Setting up RAG system...")
print("Setting up RAG system...")

# Set GCS_BUCKET_NAME for rag_helpers
os.environ["GCS_BUCKET_NAME"] = "stock-busters-chroma-bucket"
print(f"Using ChromaDB bucket: stock-busters-chroma-bucket")

# Connect to existing ChromaDB in GCS using rag_helpers
print(f"Connecting to ChromaDB in GCS bucket: {GCS_CHROMADB_BUCKET}")
get_rag_connection()

# Get the ChromaDB client and collection from rag_helpers cache
from rag_files.rag_helpers import _cache, VECTOR_COLLECTION
import tempfile

key = "default"
if key not in _cache:
    raise RuntimeError("Failed to get ChromaDB connection from rag_helpers")

chroma_client, collection_name = _cache[key]
if not collection_name:
    collection_name = VECTOR_COLLECTION
print(f"Connected to ChromaDB collection: {collection_name}")


[2/5] Setting up RAG system...
Setting up RAG system...
Using ChromaDB bucket: stock-busters-chroma-bucket
Connecting to ChromaDB in GCS bucket: ac215-chroma-bucket


Downloaded 6 files from GCS


Connected to RAG/ChromaDB (collection: stocks_rag_v1)
Connected to ChromaDB collection: stocks_rag_v1


In [7]:
# Setup RAG System - Part 2: Initialize Embeddings
print("Initializing FastEmbed embeddings (BAAI/bge-small-en-v1.5)...")

# Create a LangChain-compatible embedding wrapper for FastEmbed
class FastEmbedWrapper(Embeddings):
    """Wrapper to make FastEmbed compatible with LangChain's Embeddings interface."""
    def __init__(self, embedder):
        self.embedder = embedder
    
    def embed_documents(self, texts):
        """Embed a list of documents."""
        embeddings = []
        for text in texts:
            emb = next(self.embedder.embed(text))
            if hasattr(emb, "tolist"):
                emb = emb.tolist()
            embeddings.append(emb)
        return embeddings
    
    def embed_query(self, text):
        """Embed a single query."""
        emb = next(self.embedder.query_embed(text))
        if hasattr(emb, "tolist"):
            emb = emb.tolist()
        return emb

# Get the FastEmbed embedder (uses BAAI/bge-small-en-v1.5 by default)
fastembed_embedder = get_embedder()
embeddings = FastEmbedWrapper(fastembed_embedder)
print("✓ FastEmbed embeddings initialized (384 dimensions)")


Initializing FastEmbed embeddings (BAAI/bge-small-en-v1.5)...


✓ FastEmbed embeddings initialized (384 dimensions)


In [8]:
# Setup RAG System - Part 3: Retrieve Full .md Document from ChromaDB
print("Retrieving full .md document from ChromaDB (direct retrieval, no semantic search)...")
collection = chroma_client.get_collection(name=collection_name)

# Cache file to store the document content for fastest retrieval
cache_file = Path(".chromadb_full_doc_cache.json")
full_doc_id = None
full_doc_text = None
full_doc_metadata = None
used_cache = False

# Try to load cached document content first
if cache_file.exists():
    try:
        with open(cache_file, 'r') as f:
            cache_data = json.load(f)
            cached_collection = cache_data.get("collection_name")
            cached_content = cache_data.get("full_doc_text")
            cached_metadata = cache_data.get("full_doc_metadata")
            cached_id = cache_data.get("full_doc_id")
            
            if cached_content and cached_collection == collection_name:
                full_doc_id = cached_id
                full_doc_text = cached_content
                full_doc_metadata = cached_metadata or {}
                used_cache = True
                print(f"✓ Retrieved from cache (no ChromaDB lookup needed)")
                source = full_doc_metadata.get("source", "") if isinstance(full_doc_metadata, dict) else ""
                print(f"  Source: {source[:100]}...")
            else:
                print(f"  Cache mismatch or incomplete, will retrieve from ChromaDB...")
    except Exception as e:
        print(f"  Cache file error, will retrieve from ChromaDB...")

print(f"Cache status: {'Used' if used_cache else 'Not used'}")


Retrieving full .md document from ChromaDB (direct retrieval, no semantic search)...
✓ Retrieved from cache (no ChromaDB lookup needed)
  Source: /workspace/data/LLM-Quant_Expanded_RAG_with_context.md#full_document...
Cache status: Used


In [9]:
# Setup RAG System - Part 4: Retrieve from ChromaDB if cache missed
if not full_doc_text:
    print("Retrieving from ChromaDB (first run or cache miss)...")
    
    # Try to construct ID directly from known pattern first
    possible_ids = [
        "/workspace/data/LLM-Quant_Expanded_RAG_with_context.md#full_document::chunk_0",
        "LLM-Quant_Expanded_RAG_with_context.md#full_document::chunk_0",
    ]
    
    # Try direct ID lookup first
    for possible_id in possible_ids:
        try:
            direct_results = collection.get(
                ids=[possible_id],
                include=["documents", "metadatas"]
            )
            if direct_results.get("ids") and len(direct_results["ids"]) > 0:
                full_doc_id = possible_id
                full_doc_text = direct_results["documents"][0] if direct_results.get("documents") else None
                full_doc_metadata = direct_results["metadatas"][0] if direct_results.get("metadatas") else {}
                if full_doc_text:
                    print(f"  ✓ Found by direct ID lookup")
                    break
        except Exception:
            continue
    
    # If direct ID didn't work, try sample search
    if not full_doc_text:
        try:
            sample_results = collection.get(limit=100, include=["metadatas", "ids"])
            if sample_results.get("metadatas") and sample_results.get("ids"):
                for idx, metadata in enumerate(sample_results["metadatas"]):
                    if metadata and "source" in metadata:
                        source = metadata["source"]
                        if "#full_document" in source:
                            found_id = sample_results["ids"][idx]
                            full_doc_results = collection.get(
                                ids=[found_id],
                                include=["documents", "metadatas"]
                            )
                            if full_doc_results.get("ids") and len(full_doc_results["ids"]) > 0:
                                full_doc_id = found_id
                                full_doc_text = full_doc_results["documents"][0] if full_doc_results.get("documents") else None
                                full_doc_metadata = full_doc_results["metadatas"][0] if full_doc_results.get("metadatas") else {}
                                print(f"  ✓ Found in sample search")
                                break
        except Exception:
            pass
    
    # Fallback: search through all documents
    if not full_doc_text:
        print("  Searching through all documents...")
        all_results = collection.get(limit=10000, include=["documents", "metadatas"])
        
        if all_results.get("metadatas") and all_results.get("ids") and all_results.get("documents"):
            for idx, metadata in enumerate(all_results["metadatas"]):
                if metadata and "source" in metadata:
                    source = metadata["source"]
                    if "#full_document" in source:
                        full_doc_id = all_results["ids"][idx]
                        full_doc_text = all_results["documents"][idx]
                        full_doc_metadata = metadata
                        print(f"  ✓ Found in full search")
                        break

if not full_doc_text:
    raise RuntimeError(
        "Could not find full .md document in ChromaDB. "
        "Expected document with '#full_document' marker in source metadata."
    )

# Save document content to cache for next time
if full_doc_id and full_doc_text and not used_cache:
    try:
        with open(cache_file, 'w') as f:
            json.dump({
                "full_doc_id": full_doc_id,
                "full_doc_text": full_doc_text,
                "full_doc_metadata": full_doc_metadata or {},
                "collection_name": collection_name
            }, f)
        print(f"  ✓ Cached document content (will skip ChromaDB lookup next time)")
    except Exception:
        pass

source = full_doc_metadata.get("source", "") if isinstance(full_doc_metadata, dict) else ""
print(f"✓ Using full .md document from ChromaDB ({len(full_doc_text):,} characters)")
print(f"  Source: {source[:100]}...")


✓ Using full .md document from ChromaDB (12,473 characters)
  Source: /workspace/data/LLM-Quant_Expanded_RAG_with_context.md#full_document...


In [10]:
# Setup RAG System - Part 5: Create Retriever and LLM
# Create a retriever that returns the single full document
full_doc = Document(
    page_content=full_doc_text,
    metadata=full_doc_metadata or {"source": "LLM-Quant_Expanded_RAG_with_context.md (full document)"}
)

class FullDocChromaRetriever(BaseRetriever):
    """Retriever that returns the full .md document from ChromaDB (single embedding)."""
    document: Document
    
    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None):
        return [self.document]
    
    def invoke(self, input: str, config=None, **kwargs):
        return self._get_relevant_documents(input)

retriever = FullDocChromaRetriever(document=full_doc)
print(f"✓ RAG retriever created - using single full .md document from ChromaDB ({len(full_doc_text):,} characters)")

# Initialize LLM
print("Initializing LLM (Gemini 2.5 Flash)...")
try:
    llm = ChatVertexAI(
        model="gemini-2.5-flash",
        project="stock-busters-cs115",
    )
except Exception as e:
    print(f"Warning: ADC failed, trying with explicit credentials: {e}")
    llm = ChatVertexAI(
        model="gemini-2.5-flash",
        credentials=credentials,
        project="stock-busters-cs115",
    )
print("✓ LLM initialized")


✓ RAG retriever created - using single full .md document from ChromaDB (12,473 characters)
Initializing LLM (Gemini 2.5 Flash)...
✓ LLM initialized


In [11]:
# Setup RAG System - Part 6: Create Optimized RAG Chain
# Optimize: Use system message for static context (more efficient than sending in each prompt)
class OptimizedRAGChain:
    """Optimized RAG chain that uses system message for static context."""
    def __init__(self, llm, context_text: str):
        self.llm = llm
        self.context_text = context_text
        # Pre-format the system message with context (set once, reused for all stocks)
        self.system_message = SystemMessage(
            content=f"""You are a stock analyst. Use this knowledge base to analyze stocks:
{context_text}

Guidelines: Explain why stocks are good investments. Use context metrics. Be concise, no numbers.
Example: Attractive P/E ratio. Strong RSI momentum."""
        )
    
    def invoke(self, query: str):
        """Invoke with optimized prompt structure."""
        # User message only contains the stock-specific query (much smaller, ~100-200 chars)
        # System message contains the static context (12k chars, but may be cached/optimized by LLM)
        user_message = HumanMessage(content=f"Analyze: {query}")
        messages = [self.system_message, user_message]
        response = self.llm.invoke(messages)
        return response.content if hasattr(response, 'content') else str(response)

# Create optimized chain
optimized_chain = OptimizedRAGChain(llm, full_doc_text)
print(f"✓ Optimized RAG chain created (context in system message, ~{len(full_doc_text):,} chars)")
print(f"  Each stock query will only send ~100-200 chars (vs ~12k+ chars with old method)")

# Create a wrapper to maintain the same interface
class ChainWrapper:
    def __init__(self, optimized_chain):
        self.optimized_chain = optimized_chain
    
    def invoke(self, query: str):
        return self.optimized_chain.invoke(query)

chain = ChainWrapper(optimized_chain)
print("✓ RAG chain created successfully")
print("✓ RAG system ready")


✓ Optimized RAG chain created (context in system message, ~12,473 chars)
  Each stock query will only send ~100-200 chars (vs ~12k+ chars with old method)
✓ RAG chain created successfully
✓ RAG system ready


In [12]:
# Processing Functions
def generate_reasoning_for_stock(chain, stock_data: Dict[str, Any]) -> str:
    """Generate investment reasoning for a single stock using RAG chain."""
    try:
        # Extract key metrics for a more focused query
        symbol = stock_data.get('symbol', 'N/A')
        key_metrics = {
            'symbol': symbol,
            'signal': stock_data.get('signal', ''),
            'Hybrid_Score': stock_data.get('Hybrid_Score', ''),
            'Fundamental_Score': stock_data.get('Fundamental_Score', ''),
            'Technical_Score': stock_data.get('Technical_Score', ''),
            'roe': stock_data.get('roe', ''),
            'roic': stock_data.get('roic', ''),
            'peRatio': stock_data.get('peRatio', ''),
            'RSI_14': stock_data.get('RSI_14', ''),
            'sector': stock_data.get('sector', ''),
            'industry': stock_data.get('industry', ''),
        }
        # Ultra-concise query - only essential info
        metrics_str = f"{symbol}|Signal:{key_metrics.get('signal','')}|H:{key_metrics.get('Hybrid_Score','')}|F:{key_metrics.get('Fundamental_Score','')}|T:{key_metrics.get('Technical_Score','')}|ROE:{key_metrics.get('roe','')}|ROIC:{key_metrics.get('roic','')}|PE:{key_metrics.get('peRatio','')}|RSI:{key_metrics.get('RSI_14','')}|{key_metrics.get('sector','')}/{key_metrics.get('industry','')}"
        query = metrics_str
        reasoning = chain.invoke(query)
        return reasoning
    except Exception as e:
        print(f"Error generating reasoning: {e}")
        return f"Error: {str(e)}"

def process_single_stock(args_tuple):
    """Process a single stock - designed for parallel execution."""
    idx, row, chain = args_tuple
    try:
        stock_data = row.to_dict()
        symbol = row.get('symbol', 'N/A')
        
        stock_start = time.time()
        reasoning = generate_reasoning_for_stock(chain, stock_data)
        stock_time = time.time() - stock_start
        
        return (idx, reasoning, stock_time, symbol, None)
    except Exception as e:
        return (idx, f"Error: {str(e)}", 0, row.get('symbol', 'N/A'), str(e))

print("Processing functions defined")


Processing functions defined


In [13]:
# MAIN EXECUTION - Single Progress Bar for Entire Process
print("=" * 60)
print("Stock Reasoning Generator using RAG")
print("=" * 60)

# Ensure CSV is downloaded (if not already done)
if 'df' not in locals():
    print("Downloading CSV from GCS...")
    df = download_csv_from_gcs(INPUT_CSV, GCS_BUCKET_NAME)
    print("✓ CSV downloaded\n")

# Define workflow steps for progress tracking
total_steps = 3
workflow_steps = [
    "Processing Stocks",
    "Saving Results",
    "Uploading to GCS"
]

# Create main progress bar (0-100%)
# Allocate: 90% for processing stocks, 5% for saving, 5% for uploading
main_pbar = tqdm(total=100, desc="Overall Progress", unit="%", 
                bar_format="{l_bar}{bar}| {n}% [{elapsed}<{remaining}] {desc}",
                position=0, leave=True)

# Step 1: Process Stocks
main_pbar.set_description(f"[1/{total_steps}] {workflow_steps[0]}")

# Prepare DataFrame
if SAMPLE_SIZE:
    df_processing = df.head(SAMPLE_SIZE).copy()
    print(f"\n⚠ Processing sample of {SAMPLE_SIZE} stocks for testing")
else:
    df_processing = df.copy()
    print(f"\n⚠ Processing all {len(df_processing)} stocks. This may take 5-10 minutes.")

# Add reasoning column
df_processing["rag_reasoning"] = ""

total_rows = len(df_processing)
start_time = time.time()

# Prepare tasks for parallel processing
tasks = [(idx, row, chain) for idx, row in df_processing.iterrows()]

print(f"Using parallel processing with {MAX_WORKERS} concurrent workers...")

results = {}
errors = []

# Calculate progress: stocks processing takes 90% of total (90% stocks + 5% save + 5% upload)
stocks_progress_per_item = 90.0 / total_rows

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_idx = {executor.submit(process_single_stock, task): task[0] for task in tasks}
    
    # Process completed tasks as they finish
    for future in as_completed(future_to_idx):
        idx, reasoning, stock_time, symbol, error = future.result()
        results[idx] = (reasoning, stock_time, symbol, error)
        
        if error:
            errors.append((symbol, error))
        else:
            # Update main progress bar
            main_pbar.update(stocks_progress_per_item)
            main_pbar.set_postfix_str(f"✓ {symbol} ({len(results)}/{total_rows})", refresh=True)

# Update DataFrame with results
for idx, (reasoning, stock_time, symbol, error) in results.items():
    df_processing.at[idx, "rag_reasoning"] = reasoning

total_time = time.time() - start_time
print(f"\n✓ Completed processing {total_rows} stocks in {total_time:.1f}s (avg {total_time/total_rows:.1f}s per stock)")

if errors:
    print(f"\n⚠ Warning: {len(errors)} stocks had errors:")
    for symbol, error in errors[:5]:
        print(f"   • {symbol}: {error}")
    if len(errors) > 5:
        print(f"   ... and {len(errors) - 5} more errors")

# Step 2: Save Results
main_pbar.set_description(f"[2/{total_steps}] {workflow_steps[1]}")
main_pbar.set_postfix_str("", refresh=True)
local_output = "combined_quantamental_hybrid_with_factors_and_backtest_with_reasoning.csv"
df_processing.to_csv(local_output, index=False)
main_pbar.update(5)  # 5% for saving

# Step 3: Upload to GCS
main_pbar.set_description(f"[3/{total_steps}] {workflow_steps[2]}")
main_pbar.set_postfix_str("", refresh=True)
upload_csv_to_gcs(df_processing, OUTPUT_CSV, GCS_BUCKET_NAME)
main_pbar.update(5)  # 5% for uploading

# Complete
main_pbar.set_description("✓ Complete!")
main_pbar.set_postfix_str("", refresh=True)
main_pbar.close()

print("\n" + "=" * 60)
print("SUCCESS! Enhanced CSV uploaded to GCS")
print(f"Location: gs://{GCS_BUCKET_NAME}/{OUTPUT_CSV}")
print("=" * 60)


Stock Reasoning Generator using RAG


Overall Progress:   0%|          | 0% [00:00<?] Overall Progress

[1/3] Processing Stocks:   0%|          | 0% [00:00<?] [1/3] Processing Stocks: 


⚠ Processing all 390 stocks. This may take 5-10 minutes.
Using parallel processing with 20 concurrent workers...


[1/3] Processing Stocks:   0%|          | 0.23076923076923078% [00:03<22:22] [1/3] Processing Stocks: 

...

[3/3] Uploading to GCS: 100%|█████████▉| 99.9999999999998% [03:39<00:00] [3/3] Uploading to GCS: 

✓ Complete!: 100%|█████████▉| 99.9999999999998% [03:39<00:00] ✓ Complete!:                       

✓ Complete!: 100%|█████████▉| 99.9999999999998% [03:39<00:00] ✓ Complete!: 

✓ Complete!: 100%|█████████▉| 99.9999999999998% [03:39<00:00] ✓ Complete!: 

Successfully uploaded to gs://fin-data-bucket-115/model_output/combined_quantamental_hybrid_with_factors_and_backtest_with_reasoning.csv

SUCCESS! Enhanced CSV uploaded to GCS
Location: gs://fin-data-bucket-115/model_output/combined_quantamental_hybrid_with_factors_and_backtest_with_reasoning.csv


In [14]:
# Note: Save and Upload are now integrated into the main execution cell above
# This cell is kept for reference or can be used to re-run save/upload if needed


In [15]:
# Preview Results
print("Preview of results:")
print(f"\nTotal stocks processed: {len(df_processing)}")
print(f"Columns: {list(df_processing.columns)[-5:]}")

# Show sample reasoning
print("\n=== Sample Reasoning Outputs ===\n")
for i in range(min(5, len(df_processing))):
    row = df_processing.iloc[i]
    symbol = row['symbol']
    signal = row['signal']
    reasoning = row['rag_reasoning']
    
    print(f"[{i+1}] {symbol} - {signal}")
    print(f"    {reasoning[:200]}...")
    print()


Preview of results:

Total stocks processed: 390
Columns: ['cagr', 'equity_chart_path', 'sector', 'industry', 'rag_reasoning']

=== Sample Reasoning Outputs ===

[1] UHS - LONG-Outperform
    UHS presents a strong LONG-Outperform signal with a high overall blended strength. Its long-term business quality is robust, marked by efficient management and a strong competitive advantage. Valuatio...

[2] MPC - LONG-Outperform
    MPC shows a strong overall outlook, with excellent long-term fundamental quality and robust short-term technical momentum. Its valuation appears attractive, and the company demonstrates efficient mana...

[3] BAC - LONG-Outperform
    This stock receives a strong LONG-Outperform signal, indicating a high probability of beating the market next month. It presents an attractive valuation with a relatively low Price-to-Earnings ratio. ...

[4] HCA - LONG-Outperform
    HCA is signaled for long-term outperformance with a strong overall blended outlook. It demonstrates goo